# 05 - Local completion heads (decision gate G3)
Three variants: a pointwise head fitted post hoc on the frozen backbone, the dependency-propagating
head (reads its neighbours' previous done probabilities, like the automaton of Proposition 3), and
both trained jointly as an auxiliary task. Target: position i is done at step t if it is correct
at t and at every later step. Halting: every position reports done with probability > tau.

**G3 (pre-registered):** on OOD-depth the best completion-head variant beats the best baseline by
>= 10 accuracy points or >= 50% fewer premature stops, CI excluding zero across seeds.

In [1]:
# parameters
CONFIGS = ["configs/main_addition.yaml", "configs/main_parity.yaml", "configs/main_gates.yaml"]
TASKS = ["addition", "parity"]
SPLITS = ["id_test", "ood_depth", "ood_length", "ood_both"]

In [2]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [3]:
from glwm2.eval.results import configs_from_yaml, finished, curve_rows, step_acc
from glwm2.eval.analysis import (ENTRIES, BASELINES, OURS, POOLED_JOINT, CONF_RULES, entry_rows, entry_instances,
                                 summary_table, g2_check, g3_check, binned, premature_ci, length_leakage_ci)
from glwm2.eval.plotting import savefig, FAMILY_COLORS
from glwm2.utils.io import write_json
cfgs = {t: [c for f in CONFIGS for c in configs_from_yaml(PROJECT_ROOT / f) if c.task == t] for t in TASKS}
for t in TASKS:
    print(t, f"{len(finished(cfgs[t]))}/{len(cfgs[t])} runs finished, seeds:",
          sorted({c.seed for c in finished(cfgs[t])}))
rows = {t: entry_rows(cfgs[t]) for t in TASKS}

addition 14/14 runs finished, seeds: [0, 1]
parity 14/14 runs finished, seeds: [0, 1]


In [4]:
g3 = {}
for t in TASKS:
    for split in ["ood_depth", "ood_both"]:
        inst = entry_instances(cfgs[t], split)
        r = g3_check(inst)
        g3[f"{t}/{split}"] = r
        print(t, split, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items() if k not in ("acc_diff", "prem_diff")})
        if r:
            print("   acc diff (point, lo, hi):", np.round(r["acc_diff"], 3), "  premature diff:", np.round(r["prem_diff"], 3))
G3 = all(g3[f"{t}/ood_depth"].get("G3", False) for t in TASKS)
print("G3 on OOD-depth (pre-registered):", G3)

addition ood_depth {'best_ours': 'done_post', 'best_baseline': 'max_softmax', 'seeds': [0, 1], 'acc_ours': 1.0, 'acc_base': 1.0, 'prem_ours': 0.0, 'prem_base': 0.0, 'passes_acc': False, 'passes_premature': False, 'G3': False}
   acc diff (point, lo, hi): [0. 0. 0.]   premature diff: [0. 0. 0.]


addition ood_both {'best_ours': 'done_prop_joint', 'best_baseline': 'max_softmax', 'seeds': [0, 1], 'acc_ours': 1.0, 'acc_base': 0.998, 'prem_ours': 0.0, 'prem_base': 0.0, 'passes_acc': False, 'passes_premature': False, 'G3': False}
   acc diff (point, lo, hi): [0.002 0.    0.004]   premature diff: [0. 0. 0.]


parity ood_depth {'best_ours': 'done_prop_joint', 'best_baseline': 'act', 'seeds': [0, 1], 'acc_ours': 0.342, 'acc_base': 0.989, 'prem_ours': 1.0, 'prem_base': 0.976, 'passes_acc': False, 'passes_premature': False, 'G3': False}
   acc diff (point, lo, hi): [-0.647 -0.677 -0.616]   premature diff: [0.024 0.019 0.029]


parity ood_both {'best_ours': 'done_post', 'best_baseline': 'kl', 'seeds': [0, 1], 'acc_ours': 0.0, 'acc_base': 0.567, 'prem_ours': 1.0, 'prem_base': 0.285, 'passes_acc': False, 'passes_premature': False, 'G3': False}
   acc diff (point, lo, hi): [-0.566 -0.787 -0.344]   premature diff: [0.715 0.671 0.758]
G3 on OOD-depth (pre-registered): False


In [5]:
for t in TASKS:
    keep = OURS + ["oracle_stable", "max_softmax", "entropy", "stability", "trm_post", "act", "ponder", "trm_joint"]
    sub = rows[t][rows[t].entry.isin(keep)]
    print(f"== {t}"); display(sub.pivot_table(index="entry", columns="split", values=["acc", "premature", "mean_steps"], aggfunc="mean")
                              .reindex([e for e in ENTRIES if e in keep]).round(3))

== addition


acc                                               \
split           id_test id_val ood_both ood_depth ood_length ood_val   
entry                                                                  
oracle_stable     1.000  1.000    0.996     1.000      1.000   1.000   
max_softmax       1.000  1.000    0.998     1.000      1.000   1.000   
entropy           1.000  1.000    0.996     1.000      1.000   1.000   
stability         0.999  1.000    0.998     0.999      1.000   0.999   
trm_post          0.996  0.996    0.500     0.995      0.524   0.910   
act               0.999  0.998    0.360     0.998      0.432   0.978   
ponder            1.000  1.000    0.476     0.995      0.813   1.000   
trm_joint         1.000  1.000    0.319     1.000      0.489   1.000   
done_post         1.000  1.000    0.996     1.000      1.000   1.000   
done_prop_post    1.000  0.999    0.996     0.998      1.000   0.999   
done_joint        1.000  1.000    0.990     1.000      1.000   1.000   
done_prop_joint   1.000  1.000    1.000     1.000      1.000   1.000   

                mean_steps                                               \
split              id_test id_val ood_both ood_depth ood_length ood_val   
entry                                                                     
oracle_stable        2.874  2.757   40.332     4.881      4.398   7.505   
max_softmax          5.093  5.090   75.561    13.496      5.473  14.041   
entropy              5.093  5.090   75.365    13.496      5.472  14.041   
stability            3.849  3.750   40.474     5.227      5.398   8.408   
trm_post             3.593  3.470   50.035     5.382      3.340   9.366   
act                  2.601  2.548    2.994     2.440      2.762   3.422   
ponder               4.969  4.872    5.546     6.471      5.247   9.062   
trm_joint            2.856  2.754   15.106     4.248      2.990   7.414   
done_post            5.092  5.090   75.356    13.496      5.472  14.041   
done_prop_post       5.092  5.089   75.356    13.490      5.472  14.038   
done_joint           5.093  5.090   75.356    13.496      5.472  14.041   
done_prop_joint      5.093  5.090   75.356    13.496      5.472  14.041   

                premature                                               
split             id_test id_val ood_both ood_depth ood_length ood_val  
entry                                                                   
oracle_stable       0.488  0.505    0.491     0.730      0.304   0.556  
max_softmax         0.000  0.000    0.000     0.000      0.000   0.000  
entropy             0.000  0.000    0.000     0.000      0.000   0.000  
stability           0.432  0.446    0.497     0.746      0.253   0.556  
trm_post            0.463  0.480    0.996     0.753      0.675   0.646  
act                 0.646  0.643    1.000     1.000      0.703   1.000  
ponder              0.362  0.374    1.000     0.751      0.414   0.556  
trm_joint           0.490  0.505    1.000     0.746      0.673   0.556  
done_post           0.001  0.000    0.000     0.000      0.000   0.000  
done_prop_post      0.000  0.001    0.000     0.006      0.000   0.002  
done_joint          0.000  0.000    0.000     0.000      0.000   0.000  
done_prop_joint     0.000  0.000    0.000     0.000      0.000   0.000

== parity


acc                                               \
split           id_test id_val ood_both ood_depth ood_length ood_val   
entry                                                                  
oracle_stable     1.000  1.000    0.854     1.000      1.000   1.000   
max_softmax       1.000  1.000    0.008     0.510      1.000   0.392   
entropy           1.000  1.000    0.000     0.352      1.000   0.269   
stability         1.000  1.000    0.045     0.510      1.000   0.451   
trm_post          0.998  0.998    0.000     0.322      0.861   0.148   
act               1.000  1.000    0.007     0.989      0.862   0.974   
ponder            1.000  1.000    0.000     0.967      1.000   0.964   
trm_joint         1.000  1.000    0.002     0.909      0.810   0.720   
done_post         0.998  0.998    0.000     0.306      1.000   0.216   
done_prop_post    1.000  1.000    0.000     0.306      1.000   0.224   
done_joint        0.998  0.999    0.000     0.322      0.998   0.211   
done_prop_joint   1.000  0.999    0.000     0.342      1.000   0.252   

                mean_steps                                               \
split              id_test id_val ood_both ood_depth ood_length ood_val   
entry                                                                     
oracle_stable        3.516  3.451   90.068    11.800      4.517  12.608   
max_softmax          4.910  4.805   37.445    10.286      5.379  10.369   
entropy              4.910  4.805   17.884     9.411      5.379   9.418   
stability            4.346  4.260   54.106    10.747      5.403  11.199   
trm_post             4.553  4.453    8.546     9.071      4.917   8.630   
act                  3.523  3.456   52.353    11.376      4.344  12.074   
ponder               5.550  5.481   58.890    13.110      6.442  13.680   
trm_joint            3.516  3.451   26.972    11.277      4.260  11.733   
done_post            4.905  4.798    8.971     8.970      5.376   8.927   
done_prop_post       4.909  4.805    8.994     8.978      5.379   8.986   
done_joint           4.903  4.798    8.890     8.992      5.370   8.827   
done_prop_joint      4.908  4.802    8.970     8.996      5.378   8.949   

                premature                                               
split             id_test id_val ood_both ood_depth ood_length ood_val  
entry                                                                   
oracle_stable       0.890  0.866    0.187     0.592      0.886   0.570  
max_softmax         0.069  0.064    0.982     0.908      0.081   0.960  
entropy             0.069  0.064    1.000     0.992      0.081   1.000  
stability           0.371  0.366    0.855     0.862      0.043   0.861  
trm_post            0.409  0.414    1.000     1.000      0.382   1.000  
act                 0.888  0.862    1.000     0.976      0.886   0.994  
ponder              0.143  0.135    0.922     0.303      0.020   0.347  
trm_joint           0.890  0.866    1.000     0.926      0.886   0.966  
done_post           0.071  0.069    1.000     1.000      0.082   1.000  
done_prop_post      0.070  0.065    1.000     1.000      0.081   1.000  
done_joint          0.073  0.069    1.000     1.000      0.085   1.000  
done_prop_joint     0.071  0.066    1.000     1.000      0.082   1.000

In [6]:
# completion-head calibration (ECE of per-position done probabilities against done targets)
from glwm2.train.trainer import run_dirs
from glwm2.utils.io import read_json
cal = []
for t in TASKS:
    for c in finished(cfgs[t]):
        if c.method not in ("base", "done_joint", "done_prop_joint"): continue
        for split, d in read_json(run_dirs(c)[1] / "calibration.json").items():
            for head, ece in d.items():
                cal.append(dict(task=t, method=c.method, head=head, split=split, seed=c.seed, ece=ece))
cal = pd.DataFrame(cal)
display(cal.pivot_table(index=["task", "method", "head"], columns="split", values="ece", aggfunc="mean").round(4))

split                                    id_test  id_val  ood_both  ood_depth  \
task     method          head                                                   
addition base            done_post        0.0025  0.0014    0.0220     0.0126   
                         done_prop_post   0.0026  0.0015    0.0219     0.0126   
         done_joint      done_joint       0.0021  0.0011    0.0201     0.0137   
                         done_post        0.0023  0.0013    0.0201     0.0139   
                         done_prop_post   0.0023  0.0012    0.0209     0.0137   
         done_prop_joint done_joint       0.0021  0.0011    0.0198     0.0135   
                         done_post        0.0022  0.0011    0.0195     0.0135   
                         done_prop_post   0.0022  0.0011    0.0192     0.0135   
parity   base            done_post        0.0008  0.0006    0.1591     0.0702   
                         done_prop_post   0.0008  0.0006    0.1592     0.0704   
         done_joint      done_joint       0.0007  0.0006    0.1525     0.0704   
                         done_post        0.0007  0.0006    0.1524     0.0702   
                         done_prop_post   0.0007  0.0006    0.1524     0.0703   
         done_prop_joint done_joint       0.0007  0.0007    0.1591     0.0711   
                         done_post        0.0008  0.0006    0.1591     0.0710   
                         done_prop_post   0.0008  0.0006    0.1591     0.0711   

split                                    ood_length  ood_val  
task     method          head                                 
addition base            done_post           0.0023   0.0103  
                         done_prop_post      0.0023   0.0104  
         done_joint      done_joint          0.0019   0.0093  
                         done_post           0.0020   0.0095  
                         done_prop_post      0.0020   0.0095  
         done_prop_joint done_joint          0.0019   0.0093  
                         done_post           0.0019   0.0092  
                         done_prop_post      0.0019   0.0091  
parity   base            done_post           0.0016   0.0511  
                         done_prop_post      0.0016   0.0512  
         done_joint      done_joint          0.0014   0.0514  
                         done_post           0.0015   0.0513  
                         done_prop_post      0.0015   0.0513  
         done_prop_joint done_joint          0.0014   0.0519  
                         done_post           0.0015   0.0519  
                         done_prop_post      0.0015   0.0519

In [7]:
# H4: does joint training distort the per-step accuracy curve? AUC of ID-val accuracy over steps 1..T_max
auc = []
for t in TASKS:
    for c in finished(cfgs[t]):
        a = step_acc(c)["id_val"]
        auc.append(dict(task=t, method=c.method, seed=c.seed, auc=float(np.mean(a))))
auc = pd.DataFrame(auc)
h4 = auc.pivot_table(index="method", columns="task", values="auc", aggfunc="mean")
h4_diff = (h4 - h4.loc["base"]) * 100
display(h4.round(4)); print("difference from base, accuracy points:"); display(h4_diff.round(2))

task,addition,parity
method,,
act,0.5890,0.9057
base,0.9325,0.9057
done_joint,0.9324,0.9057
done_prop_joint,0.9325,0.9057
ponder,0.9308,0.9046
trm_joint,0.9325,0.9057
v1gate,0.7668,0.0492


difference from base, accuracy points:


task,addition,parity
method,,
act,-34.35,0.00
base,0.00,0.00
done_joint,-0.01,0.00
done_prop_joint,-0.00,0.00
ponder,-0.17,-0.11
trm_joint,0.00,0.00
v1gate,-16.56,-85.65


In [8]:
# how the done signal spreads over positions and steps (deep OOD-both addition instance)
from glwm2.eval.results import per_instance
from glwm2.data.splits import load_split
from glwm2.eval.trajectory import per_position_rho
c = next(c for c in finished(cfgs["addition"]) if c.method == "base")
pi = per_instance(c)
data = load_split("addition", "ood_both")
heads = [h for h in ["done_post", "done_prop_post"] if h in pi["pos_done"]["ood_both"]]
k = int(torch.argmax(data["depth"][:64]))
L = int(data["length"][k]); rho = per_position_rho("addition", {kk: v[k:k+1] for kk, v in data.items()})[0, :L+1].numpy()
fig, axes = plt.subplots(1, len(heads), figsize=(7.0, 2.4), squeeze=False)
for ax, h in zip(axes[0], heads):
    P = pi["pos_done"]["ood_both"][h][:, k, :L+1].float().numpy()
    im = ax.imshow(P.T, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1, extent=[0.5, P.shape[0] + 0.5, -0.5, L + 0.5])
    ax.plot(rho, np.arange(L + 1), color="#D55E00", lw=1)
    ax.set_title(f"{h}: done probability"); ax.set_xlabel("step t"); ax.set_ylabel("position i")
fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8)
WRITTEN.append(savefig(fig, "figA_done_spread"))

In [9]:
WRITTEN.append(write_json(RESULTS_DIR / "gates" / "g3.json", {"G3": G3, "detail": g3, "h4_auc_diff_points": h4_diff.to_dict()}))

In [10]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))

Files written by this notebook:
  paper\figures\figA_done_spread.pdf
  results\gates\g3.json
